# exp4 — раунд увеличения скора: больше эпох ALS, PPR-рефинемент, fusion-доводка, размер пула

**Дата:** 2026-10-23. **Seed:** 42 везде. **Контекст:** победитель exp3 —
`exp3_e3b` (RRF k=20 w=(1,2,1) + PPR E1 aw=0.5 без decay + ALS 64/0.05/20),
Kaggle 0.34487/0.32361. Цель раунда — найти конфигурацию лучше.

## Гипотезы
- **E5 («больше эпох»)**: ALS — самый слабый компонент (отдельно 0.140/0.092);
  больше итераций/факторов усилит его вклад в fusion.
- **E6**: оптимум `attr_weight` может быть выше 0.5 (в E1 ряд был монотонный);
  `restart ∈ {0.3,0.5,0.7}`. Плюс численная проверка: при alpha=1−restart
  сжатие alpha^30, итераций 30 достаточно?
- **E7**: в окрестности победителя (k, w_ppr, w_als) есть близкая точка лучше.
- **E8**: пул топ-500 урезает кандидатов; 1000/2000 дадут рост recall.

## Метод
Оба сплита (main: train≤20/val21–30; early: train≤16/val17–20), строгая
релевантность, join = канал + overlap ≥300s или ≥50%. Локальный фильтр:
не хуже победителя exp3 (0.5365/0.4632) на обоих сплитах. Решения по Kaggle.

In [ ]:
import pandas as pd
e5 = pd.read_csv("results/tables/exp4_als.csv")
print(e5.to_string(index=False))

                                                        variant                                                                           params  MAP5_main  MAP5_early  MAP5_mean  ALS_standalone_main  ALS_standalone_early  ALS_standalone_mean  time_s  seed
                                    E5 ALS f=128 it=20 reg=0.05  factors=128; iterations=20; reg=0.05; fusion=k20 w=(1.0, 2.0, 1.0); ppr=E1aw0.5   0.538200    0.461828   0.500014             0.122306              0.082704             0.102505     9.3    42
                                     E5 ALS f=64 it=40 reg=0.05   factors=64; iterations=40; reg=0.05; fusion=k20 w=(1.0, 2.0, 1.0); ppr=E1aw0.5   0.537459    0.462539   0.499999             0.138578              0.090114             0.114346     9.8    42
                                      E5 ALS f=64 it=40 reg=0.2    factors=64; iterations=40; reg=0.2; fusion=k20 w=(1.0, 2.0, 1.0); ppr=E1aw0.5   0.537934    0.462045   0.499990             0.138179              0.091400        

**E5 результат.** Таблица `exp4_als.csv`: ALS-гиперпараметры, fusion MAP@5
(к k=20 w=(1,2,1), PPR E1 фиксирован) и ALS отдельно на обоих сплитах.

In [ ]:
e6 = pd.read_csv("results/tables/exp4_ppr.csv")
print(e6.to_string(index=False))

                       variant                                                                                                       params  MAP5_main  MAP5_early  MAP5_mean  PPR_standalone_main  PPR_standalone_early  PPR_standalone_mean  time_s  seed
E6 restart=0.7 attr_weight=0.7 restart=0.7; alpha=0.3; attr_weight=0.7; decay=none; n_iter=30; fusion=k20 w=(1.0, 2.0, 1.0); als=64/0.05/20   0.544459    0.467073   0.505766             0.524201              0.439442             0.481822    21.3    42
E6 restart=0.7 attr_weight=0.6 restart=0.7; alpha=0.3; attr_weight=0.6; decay=none; n_iter=30; fusion=k20 w=(1.0, 2.0, 1.0); als=64/0.05/20   0.541808    0.469278   0.505543             0.525560              0.442814             0.484187    20.4    42
E6 restart=0.7 attr_weight=0.5 restart=0.7; alpha=0.3; attr_weight=0.5; decay=none; n_iter=30; fusion=k20 w=(1.0, 2.0, 1.0); als=64/0.05/20   0.541443    0.468453   0.504948             0.525835              0.443986             0.484911    86.

**E6 результат.** Сетка restart × attr_weight + строка референса
(0.32306/0.29313 из exp3_e1.csv). Численная проверка сходимости —
`exp4_ppr_convergence.csv`: max|score(30 iter) − score(100 iter)|.

In [ ]:
conv = pd.read_csv("results/tables/exp4_ppr_convergence.csv")
print(conv.groupby(["restart", "alpha"])["max_abs_score_diff"]
        .agg(["max", "mean"]).to_string())
print("\nalpha^30 (теоретическое сжатие):")
print(conv.groupby("restart")["contraction_alpha_pow30"].first().to_string())

                        max          mean
restart alpha                            
0.3     0.7    8.850123e-08  7.318790e-08
0.5     0.5    2.960306e-12  2.448087e-12
0.7     0.3    6.938894e-18  6.938894e-18

alpha^30 (теоретическое сжатие):
restart
0.3    2.253934e-05
0.5    9.313226e-10
0.7    2.058911e-16


**Численная проверка сходимости.** Если max|Δ| на уровне 1e-9 при alpha=0.5
(теория: 0.5^30 ≈ 9.3e-10), то 30 итераций достаточно и увеличивать эпохи PPR
бессмысленно. Для restart=0.3 (alpha=0.7) сжатие 0.7^30 ≈ 2.3e-5 — разность
должна быть больше, но по-прежнему мала для порядка рангов.

In [ ]:
e7 = pd.read_csv("results/tables/exp4_fusion.csv")
print(e7.head(10).to_string(index=False))

                                                                variant                                                                                                         params  MAP5_main  MAP5_early  MAP5_mean  time_s  seed  exp3_winner_point
                                  E7 k=10 w_pop=1.0 w_ppr=2.0 w_als=1.0 k_rrf=10; w_pop=1.0; w_ppr=2.0; w_als=1.0; ppr=E6 restart=0.7 attr_weight=0.7; als=E5 ALS f=128 it=20 reg=0.05   0.547871    0.471534   0.509703   0.050    42              False
                                  E7 k=10 w_pop=1.0 w_ppr=1.5 w_als=0.5 k_rrf=10; w_pop=1.0; w_ppr=1.5; w_als=0.5; ppr=E6 restart=0.7 attr_weight=0.7; als=E5 ALS f=128 it=20 reg=0.05   0.545023    0.471353   0.508188   0.056    42              False
                                  E7 k=10 w_pop=1.0 w_ppr=1.5 w_als=1.0 k_rrf=10; w_pop=1.0; w_ppr=1.5; w_als=1.0; ppr=E6 restart=0.7 attr_weight=0.7; als=E5 ALS f=128 it=20 reg=0.05   0.544842    0.470572   0.507707   0.052    42              False


**E7 результат.** Fusion-доводка из 18 комбинаций (k ∈ {10,20,40},
w_ppr ∈ {1.5,2,3}, w_als ∈ {0.5,1}) поверх лучших ALS/PPR из E5/E6.

In [ ]:
e8 = pd.read_csv("results/tables/exp4_pool.csv")
print(e8.to_string(index=False))

     variant                                                                                                                                                                   params  MAP5_main  MAP5_early  MAP5_mean  time_s  seed  baseline_pool500
 E8 pool=500  pool=500; ppr={'restart': 0.7, 'attr_weight': 0.7}; als={'factors': 128, 'iterations': 20, 'reg': 0.05}; fusion={'k_rrf': 10, 'w_pop': 1.0, 'w_ppr': 2.0, 'w_als': 1.0}   0.547871    0.471534   0.509703    34.6    42              True
E8 pool=1000 pool=1000; ppr={'restart': 0.7, 'attr_weight': 0.7}; als={'factors': 128, 'iterations': 20, 'reg': 0.05}; fusion={'k_rrf': 10, 'w_pop': 1.0, 'w_ppr': 2.0, 'w_als': 1.0}   0.546952    0.471593   0.509272    39.0    42             False
E8 pool=2000 pool=2000; ppr={'restart': 0.7, 'attr_weight': 0.7}; als={'factors': 128, 'iterations': 20, 'reg': 0.05}; fusion={'k_rrf': 10, 'w_pop': 1.0, 'w_ppr': 2.0, 'w_als': 1.0}   0.547256    0.470430   0.508843    41.3    42             False


**E8 результат.** Размер пула 500/1000/2000 с конфигурацией-победителем.